# NetDetect: Social Network Analysis and Influence Propagation

This notebook walks through a complete graph analysis workflow:

1. **Network Construction**: synthetic graphs with planted community structure
2. **Community Detection**: Louvain, Label Propagation, Girvan-Newman, Spectral
3. **Centrality Analysis**: Degree, Betweenness, Eigenvector, PageRank
4. **Influence Propagation**: Independent Cascade simulation and seed strategy comparison
5. **Link Prediction**: heuristic and supervised ML approaches

**Author:** Michael Yip  
**Stack:** Python, NetworkX, scikit-learn, matplotlib, seaborn


In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
%matplotlib inline

from src.network_builder import (
    generate_stochastic_block_model, generate_lfr_benchmark,
    load_karate_club, get_network_summary
)
from src.community_detection import run_all_detectors, louvain, evaluate_partition
from src.influence_analysis import (
    compute_centralities, get_top_influencers,
    independent_cascade, compare_seed_strategies
)
from src.link_prediction import (
    train_test_split_edges, evaluate_heuristics, train_link_predictor
)
from src.visualisation import (
    plot_network_communities, plot_detection_comparison,
    plot_centrality_distributions, plot_centrality_correlation,
    plot_influence_comparison, plot_feature_importances,
    plot_degree_distribution
)

print('All imports loaded')

---
## 1. Network Construction

We generate a **Stochastic Block Model (SBM)** with 5 planted communities.  
Intra-community edge probability is 25 times the inter-community probability,  
which produces separated but still noisy clusters.


In [ ]:
G = generate_stochastic_block_model(
    sizes=[60, 80, 50, 70, 40],
    p_intra=0.25,
    p_inter=0.01,
    seed=42
)

summary = get_network_summary(G)
pd.Series(summary, name='Value').to_frame()

In [ ]:
plot_degree_distribution(G)
plt.show()

---
## 2. Community Detection

We run four algorithms and evaluate against the planted ground truth:

| Algorithm | Time Complexity | Requires k? |
|---|---|---|
| Louvain | O(n log n) | No |
| Label Propagation | O(m) | No |
| Girvan-Newman | O(m²n) | Yes |
| Spectral Clustering | O(n³) | Yes |

In [ ]:
results = run_all_detectors(G, k=5)

metrics_df = pd.DataFrame({
    name: metrics for name, (_, metrics) in results.items()
}).T
metrics_df.index.name = 'Algorithm'
metrics_df

In [ ]:
plot_detection_comparison(results)
plt.show()

In [ ]:
# Visualise the best partition
best_name = max(results, key=lambda n: results[n][1]['modularity'])
best_partition = results[best_name][0]

plot_network_communities(G, best_partition, title=f'Communities ({best_name})')
plt.show()
print(f'Best algorithm: {best_name}')

---
## 3. Centrality Analysis

Who are the most **influential** nodes in the network?  
We compute four complementary centrality measures and examine their correlations.

In [ ]:
centrality_df = compute_centralities(G)
print('Top 10 nodes by PageRank:')
get_top_influencers(centrality_df, 'pagerank', top_n=10)

In [ ]:
plot_centrality_distributions(centrality_df)
plt.show()

In [ ]:
plot_centrality_correlation(centrality_df)
plt.show()

---
## 4. Influence Propagation

We simulate the **Independent Cascade** model to study how information  
spreads through the network. Then we compare seed-selection strategies  
for **influence maximisation**.


In [ ]:
# Single simulation from the top PageRank node
top_seeds = centrality_df.nlargest(5, 'pagerank').index.tolist()
sim = independent_cascade(G, top_seeds, propagation_prob=0.1)

print(f'Seed nodes:        {top_seeds}')
print(f'Total activated:   {len(sim["activated"])} / {G.number_of_nodes()}')
print(f'Network reach:     {sim["reach"]:.1%}')
print(f'Propagation steps: {sim["steps"]}')

In [ ]:
comparison = compare_seed_strategies(
    G, centrality_df,
    seed_size=5,
    propagation_prob=0.1,
    num_simulations=200
)
comparison

In [ ]:
plot_influence_comparison(comparison)
plt.show()

---
## 5. Link Prediction

Can we predict **future connections**? We hold out 15% of edges, then compare:

- **Heuristic** predictors: Common Neighbors, Jaccard, Adamic-Adar, Preferential Attachment
- **Supervised ML**: Gradient Boosting on hand-crafted node-pair features

The supervised model is trained on observed training edges and sampled non-edges. Holdout metrics are reported on edges and non-edges that were not used during training.


In [ ]:
G_train, test_pos, test_neg = train_test_split_edges(G, test_fraction=0.15)
print(f'Train edges: {G_train.number_of_edges()}')
print(f'Test positives: {len(test_pos)}  |  Test negatives: {len(test_neg)}')

In [ ]:
heuristic_df = evaluate_heuristics(G_train, test_pos, test_neg)
heuristic_df

In [ ]:
ml_result = train_link_predictor(G_train, test_pos, test_neg)
cv_mean = ml_result['cv_mean_auc']
cv_std = ml_result['cv_std_auc']
cv_text = 'not available' if cv_mean is None else f'{cv_mean:.4f} +/- {cv_std:.4f}'

print(f'Gradient Boosting holdout AUC: {ml_result["test_auc"]:.4f}')
print(f'Gradient Boosting holdout average precision: {ml_result["test_average_precision"]:.4f}')
print(f'Training CV AUC: {cv_text}')
print('
Feature importances:')
ml_result['feature_importances']


In [ ]:
plot_feature_importances(ml_result['feature_importances'])
plt.show()

---
## Summary

| Component | What to Review |
|---|---|
| Community Detection | Compare modularity, NMI, and ARI across algorithms |
| Centrality | Check whether PageRank, Eigenvector, and Betweenness identify different nodes |
| Influence Propagation | Compare centrality-based seeding against random seeding |
| Link Prediction | Compare heuristic scores with the held-out supervised model result |

### Potential Extensions

- Dynamic or temporal network analysis
- Graph Neural Networks for node classification
- Real-world follower, citation, or interaction graphs
- Overlapping community detection methods
